## Parkinson's Disease Detection

Data in `../data` folder.

### Imports

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import GridSearchCV, StratifiedGroupKFold
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer

from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

In [ ]:
pd.set_option('display.max_columns', None)

### EDA

In [ ]:
df = pd.read_csv('../data/parkinsons.data')
df.head()

In [ ]:
print(df.dtypes)

In [ ]:
df['status'].value_counts()

In [ ]:
X = df.drop(columns=['name', 'status'])
y = df['status']
# The final underscore-separated number identifies the recording, not the person.
subjects = df['name'].str.rsplit('_', n=1).str[0]
holdout = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
train_idx, test_idx = next(holdout.split(X, y, groups=subjects))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
train_subjects, test_subjects = subjects.iloc[train_idx], subjects.iloc[test_idx]
assert set(train_subjects).isdisjoint(test_subjects)

In [ ]:
column_transformer = ColumnTransformer(
    transformers=[
        ('num', Pipeline(steps=[('imputer', SimpleImputer(strategy='mean')),
                                ('scaler', StandardScaler())]), X_train.columns)
    ])

In [ ]:
search = GridSearchCV(
    estimator=Pipeline(steps=[('preprocessor', column_transformer),
                              ('classifier', GradientBoostingClassifier(random_state=42))]),
    param_grid={
        'classifier__n_estimators': [100, 200, 300, 400, 500],
        'classifier__learning_rate': [0.1, 0.2, 0.3, 0.4, 0.5],
        'classifier__max_depth': [2, 3, 4]
    },
    scoring='f1',
    cv=StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42),
    n_jobs=-1,
    verbose=1
)

In [ ]:
for train_idx, validation_idx in search.cv.split(X_train, y_train, groups=train_subjects):
    assert set(train_subjects.iloc[train_idx]).isdisjoint(train_subjects.iloc[validation_idx])

search.fit(X_train, y_train, groups=train_subjects)

In [ ]:
print(search.best_params_)

## Results

In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay

In [ ]:
y_pred = search.predict(X_test)
print(classification_report(y_test, y_pred))
print("Accuracy:", accuracy_score(y_test, y_pred))
ConfusionMatrixDisplay.from_estimator(search, X_test, y_test)

In [ ]:
import joblib

joblib.dump(search, 'parkinsons_model.joblib')